## Starting Dataset

The cleaned English AmazonHelp conversation dataset was produced in Notebook 1 from the raw Customer Support on Twitter dataset.

A subset of 397 conversations was then manually labelled using the 11-intent taxonomy defined from observed AmazonHelp support patterns.

This notebook starts from that labelled dataset and performs the remaining modelling and agent-development workflow:

- intent classification;
- baseline comparison;
- semantic retrieval;
- response generation;
- escalation;
- response evaluation;
- failure analysis.

`https://colab.research.google.com/drive/1D8eUeeNTf2KjBsEtAtY0b-Ux712UP2eZ?usp=sharing`

In [ ]:
import pandas as pd
import json

data = pd.read_csv("data/amazonhelp_cleaned_with_intents.csv")

print("Shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

print("\nLabeled rows:", data["Intent"].notna().sum())
print("Unlabeled rows:", data["Intent"].isna().sum())

data.head()

Shape: (117376, 3)

Columns:
['cleaned_customer_message', 'cleaned_amazon_response', 'Intent']

Labeled rows: 397
Unlabeled rows: 116979


,cleaned_customer_message,cleaned_amazon_response,Intent
0,how many times we have to fill the forms 23rd ...,your order details we consider it to be person...,support_followup
1,exactly i have received standard template reve...,i get your concern amol sorry for the stretch ...,support_followup
2,please tell me why i ordered a shirt using pri...,and the delivery date was the friday 13th,delivery_issue
3,account registeted as email,i m sorry about the hassle we will look into i...,technical
4,i tried 2 buy product my acc has 10k pay balan...,please share your details from the link here t...,refund_payment


# INTENT CLASSIFICATION

In [2]:
# Separate labeled and historical data

labeled_data = data[data["Intent"].notna()].copy()
historical_data = data[data["Intent"].isna()].copy()

print("Labeled data shape:", labeled_data.shape)
print("Historical data shape:", historical_data.shape)

print("\nIntent distribution:")
print(labeled_data["Intent"].value_counts())

Labeled data shape: (397, 3)
Historical data shape: (116979, 3)

Intent distribution:
Intent
delivery_issue         90
other                  57
support_followup       54
delivery_courier       42
refund_payment         38
technical              36
item_problem           23
product_information    20
prime                  19
order_change_cancel    12
Alexa                   6
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    labeled_data,
    test_size=0.20,
    random_state=42,
    #because some intents have very few examples (Alexa = 6, order_change_cancel = 12),
    #we should use a stratified split so every intent is represented proportionally in both sets.
    stratify=labeled_data["Intent"]
)

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

print("\nTrain distribution:")
print(train_data["Intent"].value_counts())

print("\nTest distribution:")
print(test_data["Intent"].value_counts())

Train shape: (317, 3)
Test shape: (80, 3)

Train distribution:
Intent
delivery_issue         72
other                  45
support_followup       43
delivery_courier       34
refund_payment         30
technical              29
item_problem           18
product_information    16
prime                  15
order_change_cancel    10
Alexa                   5
Name: count, dtype: int64

Test distribution:
Intent
delivery_issue         18
other                  12
support_followup       11
refund_payment          8
delivery_courier        8
technical               7
item_problem            5
product_information     4
prime                   4
order_change_cancel     2
Alexa                   1
Name: count, dtype: int64


##baselines

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

majority_model = DummyClassifier(
    strategy="most_frequent"
)

majority_model.fit(
    train_data["cleaned_customer_message"],
    train_data["Intent"]
)

y_pred_majority = majority_model.predict(
    test_data["cleaned_customer_message"]
)

majority_accuracy = accuracy_score(
    test_data['Intent'],
    y_pred_majority
)

print("Majority baseline accuracy:", majority_accuracy)
print("\nPredicted class:", majority_model.classes_[majority_model.class_prior_.argmax()])

Majority baseline accuracy: 0.225

Predicted class: delivery_issue


### tf-idf + Logistic regression

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# TF-IDF features
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000
)

X_train_tfidf = tfidf.fit_transform(
    train_data["cleaned_customer_message"]
)

X_test_tfidf = tfidf.transform(
    test_data["cleaned_customer_message"]
)

# Logistic Regression
tfidf_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

tfidf_model.fit(X_train_tfidf, train_data['Intent'])

# Evaluate
y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

tfidf_accuracy = accuracy_score(
    test_data['Intent'],
    y_pred_tfidf
)

print("TF-IDF + Logistic Regression accuracy:", tfidf_accuracy)

print("\nClassification Report:")
print(classification_report(test_data['Intent'], y_pred_tfidf))

TF-IDF + Logistic Regression accuracy: 0.4

Classification Report:
                     precision    recall  f1-score   support

              Alexa       0.00      0.00      0.00         1
   delivery_courier       0.36      0.50      0.42         8
     delivery_issue       0.70      0.39      0.50        18
       item_problem       0.33      0.40      0.36         5
order_change_cancel       0.00      0.00      0.00         2
              other       0.43      0.25      0.32        12
              prime       0.57      1.00      0.73         4
product_information       0.38      0.75      0.50         4
     refund_payment       0.44      0.50      0.47         8
   support_followup       0.30      0.27      0.29        11
          technical       0.25      0.29      0.27         7

           accuracy                           0.40        80
          macro avg       0.34      0.40      0.35        80
       weighted avg       0.43      0.40      0.40        80



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Load MiniLm

In [6]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
from sentence_transformers import SentenceTransformer


In [ ]:

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
).to(device)

print("Model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully


# Load BGE embedding model

In [8]:
bge_embedding_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
).to(device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Generate embeddings for train and test messages

X_train = embedding_model.encode(
    train_data["cleaned_customer_message"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

X_test = embedding_model.encode(
    test_data["cleaned_customer_message"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

y_train = train_data["Intent"]
y_test = test_data["Intent"]

print("Training embeddings shape:", X_train.shape)
print("Test embeddings shape:", X_test.shape)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Training embeddings shape: (317, 384)
Test embeddings shape: (80, 384)


In [10]:
# Generate embeddings for train and test messages

X_train_bge = bge_embedding_model.encode(
    train_data["cleaned_customer_message"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

X_test_bge = bge_embedding_model.encode(
    test_data["cleaned_customer_message"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

y_train_bge = train_data["Intent"]
y_test_bge = test_data["Intent"]

print("Training embeddings shape:", X_train_bge.shape)
print("Test embeddings shape:", X_test_bge.shape)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Training embeddings shape: (317, 768)
Test embeddings shape: (80, 768)


## classification model (MINILM(embedding) + logistic regression)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Train Logistic Regression
intent_model_LG = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
# training on MINILM embedding
intent_model_LG.fit(X_train, y_train)

# Predict on unseen test data
y_pred = intent_model_LG.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.6375

Classification Report:
                     precision    recall  f1-score   support

              Alexa       0.00      0.00      0.00         1
   delivery_courier       0.67      0.75      0.71         8
     delivery_issue       0.86      0.67      0.75        18
       item_problem       0.50      0.60      0.55         5
order_change_cancel       0.50      0.50      0.50         2
              other       0.45      0.42      0.43        12
              prime       0.67      1.00      0.80         4
product_information       0.43      0.75      0.55         4
     refund_payment       0.75      0.75      0.75         8
   support_followup       0.58      0.64      0.61        11
          technical       0.80      0.57      0.67         7

           accuracy                           0.64        80
          macro avg       0.56      0.60      0.57        80
       weighted avg       0.65      0.64      0.64        80



## classification model BGE(embedding) + logistic regression

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Train Logistic Regression
intent_model_LG = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
# training on MINILM embedding
intent_model_LG.fit(X_train_bge, y_train_bge)

# Predict on unseen test data
y_pred_bge = intent_model_LG.predict(X_test_bge)

# Evaluate
accuracy_bge = accuracy_score(y_test_bge, y_pred_bge)

print(f"Accuracy: {accuracy_bge:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_bge, y_pred_bge, zero_division=0))

Accuracy: 0.7125

Classification Report:
                     precision    recall  f1-score   support

              Alexa       0.00      0.00      0.00         1
   delivery_courier       1.00      0.75      0.86         8
     delivery_issue       0.83      0.83      0.83        18
       item_problem       0.62      1.00      0.77         5
order_change_cancel       0.17      0.50      0.25         2
              other       0.54      0.58      0.56        12
              prime       0.80      1.00      0.89         4
product_information       0.67      0.50      0.57         4
     refund_payment       1.00      0.62      0.77         8
   support_followup       0.64      0.64      0.64        11
          technical       1.00      0.71      0.83         7

           accuracy                           0.71        80
          macro avg       0.66      0.65      0.63        80
       weighted avg       0.76      0.71      0.72        80



BGE + LinearSVC Accuracy: 0.7125



In [12]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# BGE + LinearSVC
intent_model_bge_svc = LinearSVC(
    class_weight="balanced",
    random_state=42
)

# Train
intent_model_bge_svc.fit(X_train_bge, y_train_bge)

# Predict
y_pred_bge_svc = intent_model_bge_svc.predict(X_test_bge)

# Evaluate
accuracy_bge_svc = accuracy_score(y_test_bge, y_pred_bge)

print(f"BGE + LinearSVC Accuracy: {accuracy_bge_svc:.4f}")
print("\nClassification Report:")
print(
    classification_report(
        y_test_bge,
        y_pred_bge_svc,
        zero_division=0
    )
)

BGE + LinearSVC Accuracy: 0.7125

Classification Report:
                     precision    recall  f1-score   support

              Alexa       0.00      0.00      0.00         1
   delivery_courier       0.75      0.75      0.75         8
     delivery_issue       0.68      0.83      0.75        18
       item_problem       0.75      0.60      0.67         5
order_change_cancel       0.33      0.50      0.40         2
              other       0.47      0.58      0.52        12
              prime       0.67      0.50      0.57         4
product_information       0.67      0.50      0.57         4
     refund_payment       0.83      0.62      0.71         8
   support_followup       0.60      0.55      0.57        11
          technical       0.83      0.71      0.77         7

           accuracy                           0.65        80
          macro avg       0.60      0.56      0.57        80
       weighted avg       0.66      0.65      0.65        80



## Generate BGE embeddings for historical customer messages

In [ ]:
historical_embeddings_bge = bge_embedding_model.encode(
    historical_data["cleaned_customer_message"].fillna('').tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Historical embeddings shape:", historical_embeddings_bge.shape)

Batches:   0%|          | 0/3656 [00:00<?, ?it/s]

Historical embeddings shape: (116979, 768)


## Predict intents for all historical cases


In [ ]:

historical_predicted_intents = intent_model_LG.predict(
    historical_embeddings_bge
)

# Add predictions to the historical dataset
historical_data = historical_data.copy()
historical_data["Intent"] = historical_predicted_intents

print("Historical data shape:", historical_data.shape)

print("\nPredicted intent distribution:")
print(historical_data["Intent"].value_counts())

Historical data shape: (116979, 3)

Predicted intent distribution:
Intent
delivery_issue         22284
support_followup       17151
other                  16509
delivery_courier       14458
refund_payment         10380
technical               8927
item_problem            7494
product_information     6356
order_change_cancel     5906
prime                   5781
Alexa                   1733
Name: count, dtype: int64


In [ ]:
import numpy as np

def retrieve_cases(query, top_k=5):
    query_embedding = bge_embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = historical_embeddings_bge @ query_embedding

    top_idx = np.argsort(scores)[-top_k:][::-1]

    results = historical_data.iloc[top_idx].copy()
    results["similarity"] = scores[top_idx]

    return results.reset_index(drop=True)

In [ ]:
query = "my package is delayed and I still haven't received it"

results = retrieve_cases(query, top_k=5)

print(
    results[
        [
            "cleaned_customer_message",
            "cleaned_amazon_response",
            "Intent",
            "similarity"
        ]
    ].to_string(index=False)
)

            cleaned_customer_message                                                                                                             cleaned_amazon_response         Intent  similarity
still have yet to receive my package                                                        please allow us to look into this further use this link so that we may do so delivery_issue    0.924292
     my package still hasn t arrived                                               oh no hate to hear this steve please contact us here so we can look into this for you delivery_issue    0.917025
     still haven t gotten my package  i m sorry your package hasn t arrived can you tell us when it was expected and what the current tracking shows when you check here delivery_issue    0.916076
         still waiting on my package we d like to take a closer look at this with you please use the link provided by sh to reach out to us at your earliest convenience delivery_issue    0.913507
       i still don t

## llm response

In [ ]:
#https://console.groq.com/keys
import os
os.environ["GROQ_API_KEY"] = "ENTER YOUR GROK API"

In [ ]:
!pip install -qU langchain-groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.5 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    max_tokens=200,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,

    # other params...
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

response_prompt = ChatPromptTemplate.from_template("""
You are an Amazon customer support agent.

Your task is to write a helpful reply to the customer's message.

IMPORTANT RULES:
1. Use ONLY the historical Amazon support responses provided below as evidence.
2. Do not invent policies, procedures, refunds, timelines, links, or guarantees.
3. Do not claim that you checked the customer's account or order.
4. You may combine and paraphrase information from multiple historical responses.
5. If the historical responses do not provide a clear resolution, politely ask the customer to contact Amazon support for further assistance.
6. Keep the reply concise, professional, and empathetic.
7. Do not mention these historical examples in your reply.

Customer message:
{customer_message}

Historical support examples:
{historical_examples}

Write only the final customer-facing reply.
""")

response_chain = response_prompt | llm

In [ ]:
def generate_response(query, top_k=5):

    # 1. Predict intent using trained classifier
    query_embedding = bge_embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    predicted_intent = intent_model_LG.predict(
        query_embedding.reshape(1, -1)
    )[0]

    # 2. Retrieve historical cases
    results = retrieve_cases(query, top_k=top_k)

    # 3. Prepare historical evidence
    historical_examples = "\n\n".join(
        [
            f"Customer: {row['cleaned_customer_message']}\n"
            f"Amazon response: {row['cleaned_amazon_response']}"
            for _, row in results.iterrows()
        ]
    )

    # 4. Generate response
    response = response_chain.invoke({
        "customer_message": query,
        "historical_examples": historical_examples
    })

    return predicted_intent, results, response.content

In [ ]:
query = "my package is delayed and I still haven't received it"

predicted_intent, retrieved_cases, reply = generate_response(query)

print("Predicted Intent:", predicted_intent)

print("\n" + "="*80)
print("RETRIEVED HISTORICAL CASES")
print("="*80)

for i, row in retrieved_cases.iterrows():
    print(f"\nCase {i+1}")
    print("Similarity:", round(row["similarity"], 4))
    print("Intent:", row["Intent"])
    print("Customer:", row["cleaned_customer_message"])
    print("Amazon:", row["cleaned_amazon_response"])

print("\n" + "="*80)
print("GENERATED RESPONSE")
print("="*80)
print(reply)

Predicted Intent: delivery_issue

RETRIEVED HISTORICAL CASES

Case 1
Similarity: 0.9243
Intent: delivery_issue
Customer: still have yet to receive my package
Amazon: please allow us to look into this further use this link so that we may do so

Case 2
Similarity: 0.917
Intent: delivery_issue
Customer: my package still hasn t arrived
Amazon: oh no hate to hear this steve please contact us here so we can look into this for you

Case 3
Similarity: 0.9161
Intent: delivery_issue
Customer: still haven t gotten my package
Amazon: i m sorry your package hasn t arrived can you tell us when it was expected and what the current tracking shows when you check here

Case 4
Similarity: 0.9135
Intent: delivery_issue
Customer: still waiting on my package
Amazon: we d like to take a closer look at this with you please use the link provided by sh to reach out to us at your earliest convenience

Case 5
Similarity: 0.8987
Intent: delivery_issue
Customer: i still don t have my package
Amazon: i m sorry that 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

escalation_prompt = ChatPromptTemplate.from_template("""
You are deciding whether an Amazon customer-support message can be safely
handled automatically using historical support evidence.

Customer message:
{customer_message}

Predicted intent:
{intent}

Historical support evidence:
{historical_examples}

Decision rules:

AUTO_HANDLE:
Choose this when the historical responses provide a clear and applicable
resolution or next step that can be communicated without accessing the
customer's account, order, payment, or other private information.

ESCALATE:
Choose this when resolving the issue requires account/order/payment-specific
investigation, human intervention, or information that is not available in
the customer message and historical evidence.

Do not make assumptions about the customer's account.

Return ONLY valid JSON in exactly this format:

{{
  "decision": "AUTO_HANDLE" or "ESCALATE",
  "reason": "brief explanation of why this decision was made"
}}
""")

escalation_chain = escalation_prompt | llm

In [ ]:
def decide_escalation(query, predicted_intent, retrieved_cases):

    historical_examples = "\n\n".join(
        [
            f"Customer: {row['cleaned_customer_message']}\n"
            f"Amazon response: {row['cleaned_amazon_response']}"
            for _, row in retrieved_cases.iterrows()
        ]
    )

    decision = escalation_chain.invoke({
        "customer_message": query,
        "intent": predicted_intent,
        "historical_examples": historical_examples
    })

    return decision.content

In [ ]:
result = decide_escalation(
    "my package is delayed and I still haven't received it",
    "delivery_issue",
    retrieve_cases("my package is delayed and I still haven't received it")
)

print(result)

{
  "decision": "ESCALATE",
  "reason": "The historical evidence indicates that resolving this issue requires investigating the specific order status, tracking information, or account details, which necessitates human intervention or access to private customer data not available in the message."
}


In [ ]:
decision = decide_escalation(
    query,
    predicted_intent,
    retrieved_cases
)

print("Predicted Intent:", predicted_intent)
print("Escalation Decision:", decision)

Predicted Intent: delivery_issue
Escalation Decision: ESCALATE


## LLM JUDGE

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

judge_prompt = ChatPromptTemplate.from_template("""
You are an evaluator for an AI customer-support agent.

Evaluate the generated response using ONLY the customer message and
historical support evidence provided below.

Customer message:
{customer_message}

Historical support evidence:
{historical_examples}

Generated response:
{generated_response}

Score each criterion from 1 to 5.

1. resolution_relevance:
Does the response address the customer's actual problem?

2. grounding:
Are the claims and guidance supported by the historical support evidence?

3. helpfulness:
Does the response provide a useful and appropriate next step?

4. unsupported_claims:
Does the response avoid unsupported claims, invented policies,
guarantees, or pretending to access the customer's account/order?

5. overall:
Overall quality of the support response.

Return ONLY valid JSON in exactly this format:

{{
  "resolution_relevance": <1-5>,
  "grounding": <1-5>,
  "helpfulness": <1-5>,
  "unsupported_claims": <1-5>,
  "overall": <1-5>,
  "reason": "<brief explanation max 100 words>"
}}
""")

judge_chain = judge_prompt | llm

print("Judge chain ready")

Judge chain ready


In [ ]:
historical_examples = "\n\n".join(
    [
        f"Customer: {row['cleaned_customer_message']}\n"
        f"Amazon response: {row['cleaned_amazon_response']}"
        for _, row in retrieved_cases.iterrows()
    ]
)

judge_result = judge_chain.invoke({
    "customer_message": query,
    "historical_examples": historical_examples,
    "generated_response": reply
})

print(judge_result.content)

{
  "resolution_relevance": 5,
  "grounding": 5,
  "helpfulness": 4,
  "unsupported_claims": 5,
  "overall": 5,
  "reason": "The response directly addresses the customer's complaint about the delayed package. It is well-grounded in the historical evidence, specifically mirroring the request for expected arrival dates and tracking status found in previous interactions. It avoids unsupported claims or pretending to have access to the account. The next step is appropriate for gathering necessary information to investigate the issue."
}


In [ ]:

judge_scores = json.loads(judge_result.content)

print(judge_scores)

{'resolution_relevance': 5, 'grounding': 5, 'helpfulness': 4, 'unsupported_claims': 5, 'overall': 5, 'reason': "The response directly addresses the customer's complaint about the delayed package. It is well-grounded in the historical evidence, specifically mirroring the request for expected arrival dates and tracking status found in previous interactions. It avoids unsupported claims or pretending to have access to the account. The next step is appropriate for gathering necessary information to investigate the issue."}


In [ ]:
print("Resolution relevance:", judge_scores["resolution_relevance"])
print("Grounding:", judge_scores["grounding"])
print("Helpfulness:", judge_scores["helpfulness"])
print("Unsupported claims:", judge_scores["unsupported_claims"])
print("Overall:", judge_scores["overall"])

In [ ]:
def evaluate_response(query, top_k=5):

    # 1. Predict intent
    query_embedding = bge_embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    predicted_intent = intent_model_LG.predict(
        query_embedding.reshape(1, -1)
    )[0]

    # 2. Retrieve historical cases
    retrieved_cases = retrieve_cases(query, top_k=top_k)

    # 3. Generate response
    historical_examples = "\n\n".join(
        [
            f"Customer: {row['cleaned_customer_message']}\n"
            f"Amazon response: {row['cleaned_amazon_response']}"
            for _, row in retrieved_cases.iterrows()
        ]
    )

    response = response_chain.invoke({
        "customer_message": query,
        "historical_examples": historical_examples
    }).content

    # 4. Decide AUTO_HANDLE / ESCALATE
    escalation_result = decide_escalation(
        query,
        predicted_intent,
        retrieved_cases
    )

    # 5. Judge the generated response
    judge_result = judge_chain.invoke({
        "customer_message": query,
        "historical_examples": historical_examples,
        "generated_response": response
    }).content

    # 6. Return everything
    return {
        "query": query,
        "intent": predicted_intent,
        "retrieved_cases": retrieved_cases,
        "response": response,
        "escalation": escalation_result,
        "judge_scores": json.loads(judge_result)
    }

## agent-quality evaluation:

In [ ]:
judge_test_sample = test_data.sample(
    n=10,
    random_state=123
).reset_index(drop=True)

for i, row in judge_test_sample.iterrows():
    print(f"\n{'='*80}")
    print(f"Example {i+1}")
    print("Intent:", row["Intent"])
    print("Customer:", row["cleaned_customer_message"])


Example 1
Intent: delivery_issue
Customer: two days past my guaranteed delivery date for an amazon order both of which the tracking has indicated they were out for delivery

Example 2
Intent: delivery_courier
Customer: your amazon carrier services messed up a delivery this is the 3rd time get your act together

Example 3
Intent: other
Customer: i am sorry to say this but you and can both go and fcuk yourselves this is for the nth time

Example 4
Intent: other
Customer: everytime i place a order at amazon this thing happens please rectify this

Example 5
Intent: delivery_courier
Customer: amazon package wasn t delivered because they couldn t find my address but i ve been using that address on my orders for years lmao

Example 6
Intent: Alexa
Customer: alexa what s the weather like currently in kozhikode kansas it is for fuck s sake i live in kozhikode but kozhikode is not in kansas is alexa actually intelligent you tell me she s your kid

Example 7
Intent: other
Customer: can you someh

In [ ]:
judge_examples = []

for i, row in judge_test_sample.iterrows():
    result = evaluate_response(
        row["cleaned_customer_message"],
        top_k=5
    )

    judge_examples.append(result)

    print(f"\n{'='*80}")
    print(f"Example {i+1}")
    print("True Intent:", row["Intent"])
    print("Predicted Intent:", result["intent"])
    print("Customer:", row["cleaned_customer_message"])
    print("Generated Response:", result["response"])
    print("LLM Judge Overall:", result["judge_scores"]["overall"])


Example 1
True Intent: delivery_issue
Predicted Intent: delivery_issue
Customer: two days past my guaranteed delivery date for an amazon order both of which the tracking has indicated they were out for delivery
Generated Response: I am sorry to hear that your orders have not arrived despite the tracking showing them as "out for delivery" for two days. To help us investigate this further, could you please let us know if these orders were fulfilled by Amazon or by a third-party seller? Additionally, please confirm which carrier is handling the delivery.
LLM Judge Overall: 4

Example 2
True Intent: delivery_courier
Predicted Intent: delivery_courier
Customer: your amazon carrier services messed up a delivery this is the 3rd time get your act together
Generated Response: I am very sorry to hear that you have experienced repeated delivery issues; this is certainly not the experience we want our customers to have. To help us take a closer look at what is going on, could you please provide t

In [ ]:
import pandas as pd

evaluation_rows = []

for i, result in enumerate(judge_examples):
    scores = result["judge_scores"]

    evaluation_rows.append({
        "example": i + 1,
        "intent": result["intent"],
        "resolution_relevance": scores["resolution_relevance"],
        "grounding": scores["grounding"],
        "helpfulness": scores["helpfulness"],
        "unsupported_claims": scores["unsupported_claims"],
        "overall": scores["overall"]
    })

evaluation_df = pd.DataFrame(evaluation_rows)

score_columns = [
    "resolution_relevance",
    "grounding",
    "helpfulness",
    "unsupported_claims",
    "overall"
]

print("\nAverage Scores:")
print(evaluation_df[score_columns].mean().round(2))


Average Scores:
resolution_relevance    4.0
grounding               4.7
helpfulness             3.8
unsupported_claims      5.0
overall                 4.0
dtype: float64


In [ ]:
result = evaluate_response("hey i havent recive my pacakge", 5)

print("Intent:", result["intent"])
print("\nResponse:", result["response"])
print("\nEscalation:", result["escalation"])
print("\nJudge:", result["judge_scores"])

Intent: delivery_issue

Response: I am sorry to hear that you have not received your package. To help us look into this, please share your order details here so we can check the status and assist you further.

Escalation: {
  "decision": "ESCALATE",
  "reason": "The customer reports a missing package, which requires checking specific order status and tracking information. The historical evidence shows that agents needed to access account details or ask for specific identifiers to investigate, indicating that a generic response without account access is insufficient."
}

Judge: {'resolution_relevance': 5, 'grounding': 5, 'helpfulness': 4, 'unsupported_claims': 5, 'overall': 5, 'reason': "The response directly addresses the customer's issue of not receiving a package. It is well-grounded in the historical evidence, which repeatedly asks for details to check the status. It avoids unsupported claims or pretending to have access to the account. The next step (sharing details) is appropriate